In [1]:
import torch
import transformers
print("Torch version:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

c:\Users\sthem\anaconda3\envs\swin_transformer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.5.1+cu121
CUDA disponível: True


In [2]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import random

# Caminhos para os dados
DATASET_PATH = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\cell_images"
INFECTED_PATH = os.path.join(DATASET_PATH, "Parasitized")  # Células infectadas
UNINFECTED_PATH = os.path.join(DATASET_PATH, "Uninfected")  # Células saudáveis

# Transformações de imagem para Swin Transformer
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Tamanho esperado pelo Swin Transformer
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Criar dataset personalizado
class MalariaDataset(Dataset):
    def __init__(self, data_path, transform=None):
        self.data_path = data_path
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Carregar imagens infectadas (1)
        for img_name in os.listdir(INFECTED_PATH):
            self.images.append(os.path.join(INFECTED_PATH, img_name))
            self.labels.append(1)
        
        # Carregar imagens não infectadas (0)
        for img_name in os.listdir(UNINFECTED_PATH):
            self.images.append(os.path.join(UNINFECTED_PATH, img_name))
            self.labels.append(0)

        # Embaralhar os dados
        temp = list(zip(self.images, self.labels))
        random.shuffle(temp)
        self.images, self.labels = zip(*temp)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

# Criar instâncias do dataset
dataset = MalariaDataset(DATASET_PATH, transform=transform)

# Dividir em treino e teste (80% treino, 20% teste)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Tamanho do dataset: {len(dataset)}")
print(f"Imagens de treino: {len(train_dataset)}, Imagens de teste: {len(test_dataset)}")


Tamanho do dataset: 27558
Imagens de treino: 22046, Imagens de teste: 5512


In [3]:
from transformers import SwinForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
import os
from PIL import Image

DATASET_PATH = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\cell_images"

# Listar arquivos que podem ser problemáticos
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        try:
            with Image.open(file_path) as img:
                img.verify()  # Verifica se a imagem é válida
        except Exception as e:
            print(f"Erro no arquivo: {file_path} - {str(e)}")


In [5]:
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        ext = os.path.splitext(file)[-1].lower()
        if ext not in [".png", ".jpg", ".jpeg"]:
            print(f"Arquivo inválido detectado: {file}")

In [6]:
from torchvision.models.swin_transformer import swin_t
import torchvision.models as models

def get_swin_model():
    model = swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
    num_features = model.head.in_features
    model.head = nn.Linear(num_features, 2)  # 2 classes
    return model

### Treinando o modelo

In [8]:

import optuna
from sklearn.metrics import f1_score
import torch.nn as nn
import torch.optim as optim

# Função de treino simplificada
def train_one_epoch(model, optimizer, criterion, loader, trial):
    model.train()
    loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        # trial.report(loss, "mudar este label loss")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss

# Função de avaliação: retorna F1-score
def evaluate(model, loader, trial):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    score = f1_score(all_labels, all_preds)
    # trial.report(score, "mudar este label")
        # Handle pruning based on the intermediate value.
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()
    return score

def evaluate_metrics(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    return accuracy, precision, recall, f1, tp, fp, tn, fn

# Função objetivo para o Optuna
def objective(trial):
    # Sugerir hiperparâmetros
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    lr = trial.suggest_float("lr", 1e-6, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    #optimizer_type = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])
    loss_fn = trial.suggest_categorical("loss_fn", ["CrossEntropy", "BCE"])
    
    
    # Estrutura do modelo base
    #model = SwinForImageClassification.from_pretrained("microsoft/swin-tiny-patch4-window7-224", num_labels=2, ignore_mismatched_sizes=True)
    model = get_swin_model()
    
    # Carregar pesos salvos localmente
    model_path= r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\swin-variations\swin-sem-optuna\modelos_salvos_swin\swin_fold1.pth"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])  # for hp tuning
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    
    # Treinar por poucas épocas (ajustar conforme o tempo disponível)
    for epoch in range(3):  # pode aumentar se quiser mais estabilidade
        #monitora a loss
        loss = train_one_epoch(model, optimizer, criterion, train_loader, trial)
    
    # Avaliar
    f1 = evaluate(model, test_loader, trial)
    return f1



study = optuna.create_study(direction="maximize")  # 'maximize' because objective function is returning accuracy
# study = optuna.create_study(direction="minimize")  # 'minimize' because objective function is returning loss
study.optimize(objective, n_trials=30)

pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))


[I 2025-04-01 14:34:49,083] A new study created in memory with name: no-name-e9d6a2bc-c157-4411-a4ed-d7fbf6c7ac54
C:\Users\sthem\AppData\Local\Temp\ipykernel_36240\549330364.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub f

Study statistics: 
  Number of finished trials:  30
  Number of pruned trials:  0
  Number of complete trials:  30
Best trial:
  Value:  0.9899821109123434
  Params: 
    lr: 9.597089396219724e-06
    weight_decay: 1.682757828554389e-06
    loss_fn: CrossEntropy
    optimizer: RMSprop
